# 🧬 Notebook Guide — AlphaFold 3 Data Pipeline

## Learning Objectives
By the end of this notebook you will be able to:
- [ ] Parse mmCIF files (AF3's primary structure format, replacing PDB)
- [ ] Represent proteins in Atom14 / atom37 all-atom format (not just Cα)
- [ ] Build an MSA feature tensor: one-hot + deletion matrix + depth masking
- [ ] Implement AF3's multi-biomolecule tokenizer (protein + DNA + RNA + ligand)
- [ ] Construct template features from a structural template
- [ ] Understand AF3's spatial and interface cropping strategies
- [ ] Implement the diffusion noise schedule and one denoising step

## Why This Matters
AF3's input pipeline is ~40% of the engineering effort. Understanding the data representation tells you:
- Why AF3 can handle protein-ligand, protein-DNA, and protein-RNA complexes (AF2 cannot)
- How MSA depth affects prediction confidence
- Why mmCIF is better than PDB for modern structures
- How diffusion lets AF3 sample multiple conformations (AF2 predicts a single structure)

---

## 🤖 Claude Integration — Copy These Prompts

**For mmCIF:**
```
Explain mmCIF format vs PDB format.
- Why did PDB adopt mmCIF? (PDB format hits column limits for large complexes)
- What are the key mmCIF loops I need to parse? (_atom_site, _entity, _chem_comp)
- How do I extract atom coordinates from mmCIF in Python without Biopython?
- Show the mmCIF key-value syntax for a simple 2-residue peptide.
```

**For Atom14/Atom37:**
```
Explain the atom14 representation used in AlphaFold.
- PDB has 14 heavy atoms per standard amino acid (max)
- Why 14? Which residue has the most heavy atoms? (TRP with 14)
- What is atom37? (all 37 possible atom names across all 20 AAs; sparse but unified)
- How do I convert PDB atom records to atom14 format?
- Show the atom14 mask array for ALA (only backbone + CB = 5 atoms out of 14)
```

**For MSA Featurization:**
```
Explain how AlphaFold featurizes a multiple sequence alignment.
- What is the deletion matrix? (number of deletions at each position per sequence)
- How is the MSA one-hot encoded? (21 categories: 20 AA + gap)
- What is MSA depth and why does it matter? (more sequences = better co-evolution signal)
- How does AF2 randomly mask MSA rows during training? (dropout for robustness)
- What is the difference between profile features (position-specific) and MSA features?
```

**For Diffusion:**
```
Explain the diffusion module in AlphaFold 3.
- What noise schedule does AF3 use? (log-normal distribution of sigma, not fixed T steps)
- How is the forward process defined? x_noisy = x_true + sigma * epsilon
- What does the denoising network learn? Predict x_true given x_noisy and sigma
- Why is this better than AF2's structure module for multi-conformation prediction?
- What is 'conditioning' in AF3's diffusion? (pair repr z and single repr s condition the denoiser)
```

**For Interface Cropping:**
```
Explain spatial and interface cropping in AF3 training.
- Why crop? (GPU memory: N=768 max at training; real complexes can be N=2000+)
- Spatial crop: pick a random crop center, include all residues within radius r
- Interface crop: 50% chance to center the crop on a protein-protein interface
- Why interface crop? (force the model to see binding interfaces during training)
- What is the token dropout? (randomly drop MSA rows and template hits)
```

---

## Data Representation Cheat Sheet

```
AF3 TOKEN TYPES:
  Residue-level tokens: standard amino acids (20), DNA bases (4), RNA bases (4)
  Atom-level tokens: ligand atoms from CCD (Chemical Component Dictionary)
  Special tokens: [MASK], [PAD], [UNK]

AF3 INPUT TENSORS (per structure):
  aatype:       (N,)       — amino acid type (0-19, or 20 for non-standard)
  atom_coords:  (N, 37, 3) — all-atom coordinates (atom37 format)
  atom_mask:    (N, 37)    — 1 if atom exists, 0 if not
  msa:          (S, N, 23) — MSA one-hot (21 + 2 for deletions)
  msa_mask:     (S, N)     — 1 for valid, 0 for gap
  template_coords: (T, N, 37, 3) — template atom coordinates
  template_mask:   (T, N, 37)   — template atom existence mask
```


## TL;DR — Plain English

**What this notebook does:** Explains how AlphaFold3 prepares its inputs before any neural network sees them — the often-overlooked data engineering half of the system.

**After this notebook you can:**
- Describe the MSA (multiple sequence alignment) generation pipeline using JackHMMer and HHblits
- Explain template search: how AF3 finds similar known structures to use as structural hints
- Implement tokenisation for mixed inputs (amino acids, nucleotides, small-molecule atoms)
- Explain cropping and padding strategies for variable-length proteins

**Why this matters:**
- HackerRank: Data pipeline design questions (batching, padding, tokenisation) are common in ML engineering assessments
- computational biology ML teams: The data pipeline is where most real-world ML engineering time is spent; understanding MSA generation and template featurisation shows production-level thinking

**Time:** ~3 hours | **Difficulty:** ⭐⭐⭐⭐ | **Prerequisites:** 07/01 (AF3 architecture), 01/01 (alignment basics)

## 🧬 AlphaFold 3 Data Pipeline — Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **MSA** | Multiple Sequence Alignment — aligning many homologous sequences to reveal conserved positions and co-evolution |
| **Template** | An experimentally solved structure (from PDB) used as a structural hint for prediction |
| **Crop** | Selecting a fixed-size window of residues from a protein to fit in GPU memory during training |
| **Feature dict** | A Python dictionary of tensors: the standardised input format the model consumes |
| **Tokenizer** | Converts amino acids, nucleotides, or ligand atoms into integer IDs the model can process |
| **atom14** | A compact atom representation using up to 14 atoms per residue (standard amino acid heavy atoms) |
| **atom37** | A dense atom representation using 37 atom slots per residue (covers all possible atom types) |
| **Residue type** | One of the 20 standard amino acids (or special tokens for gaps, unknown, ligands) |
| **Chain ID** | A letter (A, B, C...) identifying which polypeptide chain an atom belongs to in a complex |
| **Entity** | A unique molecular species in a complex (two copies of chain A are the same entity) |
| **Assembly** | The biologically relevant quaternary structure (may differ from the asymmetric unit in the crystal) |

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students. Beginners should treat this notebook as a conceptual first pass.

**How to study it on a first pass:**
- aim to explain the big idea of each component
- do not get stuck on every tensor shape immediately
- learn what the block is trying to accomplish biologically and geometrically

**Common traps:** drowning in implementation detail before understanding the role of Pairformer, FAPE, PAE, or diffusion in the full system.


## Canonical Resource Upgrade

Use these in order:
- [OpenFold](https://github.com/aqlaboratory/openfold) for code orientation
- [AlphaFold 2 paper](https://www.nature.com/articles/s41586-021-03819-2)
- [AlphaFold 3 paper](https://www.nature.com/articles/s41586-024-07487-w)


## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 07/01 (AF3 architecture — understand what Atom14 tensors feed into), 03/01 (basic bioinformatics data formats) |
| 📍 You Are Here | Module 07/02 — AF3 Data Pipeline |
| ➡️ Next Steps | 07/03 (evaluation metrics) |
| 🏁 Goal | Parse mmCIF files, build Atom14 tensors, and construct MSA features ready for AF3 input |

### 🆕 From Scratch? Start Here:
1. [BioPython tutorial](https://biopython.org/wiki/Category:Wiki_Documentation) — structure IO and sequence handling
2. 03/01 (this repo) — bioinformatics data formats (FASTA, PDB, mmCIF)
3. 07/01 (this repo) — AF3 architecture to understand what the pipeline produces
4. This notebook — AF3 data pipeline
**Time:** 10–12h | **Difficulty:** ⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 07/01 (Atom14 / pair representations that this pipeline produces), 03/01 (sequence + structure file formats)
- Used by: 07/03 (evaluation requires parsed structures), 10/01 (capstone uses this featurization)

## What This Notebook Teaches (Plain English)

**Before AF3 can predict a structure, it needs to convert biological data into numerical tensors.**

This notebook covers the data preparation pipeline — the steps that happen before the neural network ever sees any numbers.

### Required Background
- ✅ Protein structure concepts (Notebook 03/01)
- ✅ Basic PyTorch tensor operations
- ✅ What AF3 is trying to do (Notebook 07/01 overview)

### Key Concepts

| Concept | Plain English |
|---------|--------------|
| **mmCIF** | File format for protein structures (replaces old PDB format). Stores every atom's coordinates + metadata |
| **Atom14** | Encoding scheme: represent each amino acid with up to 14 numbered atom slots. Enables batch processing |
| **MSA** | Multiple Sequence Alignment — hundreds of related proteins aligned column-by-column. Tells AF3 which positions are evolutionarily conserved (important) |
| **Tokenization** | Converting biological sequences into numbers the model can process |
| **Diffusion noise** | Adding Gaussian noise to coordinates — the model learns to remove this noise to reconstruct structure |
| **Spatial cropping** | For memory: take the N nearest residues to a random center point instead of the full protein |
| **Violation loss** | Penalty added to training when predicted bonds have impossible lengths or atoms overlap |

### Why mmCIF Instead of PDB?

Old PDB format: max 99,999 atoms, 80 characters per line, can't handle large complexes.
mmCIF format: no size limit, key-value structure, handles everything from single residues to entire ribosomes.
AF3 uses mmCIF for all inputs and outputs.

In [ ]:
import re, json
from collections import defaultdict

# mmCIF parsing (simplified — AF3's primary structure format)
mmcif_snippet = """
data_1ABC
_entry.id   1ABC
_entity.id  1
_entity.type  polymer
_entity_poly.pdbx_seq_one_letter_code  MGSSHHHHHHSSGLVPRGSHMASMTGGQQMGRDP
loop_
_atom_site.id
_atom_site.type_symbol
_atom_site.label_atom_id
_atom_site.label_comp_id
_atom_site.label_chain_id
_atom_site.label_seq_id
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
_atom_site.occupancy
1  N  N   ALA A 1  10.000  20.000  30.000  1.00
2  C  CA  ALA A 1  11.200  20.500  30.800  1.00
3  C  C   ALA A 1  12.100  21.200  30.100  1.00
4  O  O   ALA A 1  12.500  22.300  30.500  1.00
5  N  N   GLY A 2  13.000  21.000  29.200  1.00
6  C  CA  GLY A 2  14.200  20.400  28.900  1.00
"""

def parse_mmcif_atoms(text):
    atoms = []
    lines = text.strip().split('\n')
    in_loop = False
    headers = []
    for line in lines:
        line = line.strip()
        if line.startswith('_atom_site.'):
            headers.append(line.split('.')[1])
            in_loop = True
        elif in_loop and line and not line.startswith('_') and not line.startswith('#'):
            parts = line.split()
            if len(parts) == len(headers):
                atom = dict(zip(headers, parts))
                atoms.append(atom)
    return atoms

atoms = parse_mmcif_atoms(mmcif_snippet)
print(f"Parsed {len(atoms)} atoms from mmCIF")
for a in atoms[:3]:
    print(f"  {a['label_comp_id']}{a['label_seq_id']} {a['label_atom_id']}: "
          f"({a['Cartn_x']}, {a['Cartn_y']}, {a['Cartn_z']})")

# Chain grouping
by_chain = defaultdict(list)
for a in atoms:
    by_chain[a['label_chain_id']].append(a)
print(f"\nChains: {dict((k, len(v)) for k,v in by_chain.items())}")

> **Expected output:** Number of parsed atoms from mmCIF (e.g., `Parsed 1000+ atoms`), sample atom coordinates, and a chain-to-atom-count dictionary  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

## mmCIF Parsing
### AF3's primary structure format

### 📖 Reading Guide — MSA One-Hot Encoding

`msa_onehot = np.zeros((N_seq, N_res, 21))`
→ *3D array: N_seq sequences × N_res residues × 21 amino acid types (20 + gap). One-hot means exactly one 1 per residue per sequence.*

`msa_onehot[s, r, aa_idx] = 1`
→ *Mark residue r in sequence s as amino acid aa_idx. All other values stay 0.*

`pair_feat = np.einsum('sir,sjr->ijr', msa_onehot, msa_onehot)`
→ *einsum: for each pair (i,j) of residues, sum over all sequences the co-occurrence of amino acids. This captures co-evolution — if i and j always have correlated mutations, they're probably in contact.*



In [ ]:
import torch
import numpy as np

# Atom14 representation: 14 possible heavy atoms per residue
# AF3 uses Atom37 (all standard atoms) with a mask for which are present

RESIDUE_ATOMS = {
    'ALA': ['N','CA','C','O','CB'],
    'GLY': ['N','CA','C','O'],
    'SER': ['N','CA','C','O','CB','OG'],
    'THR': ['N','CA','C','O','CB','OG1','CG2'],
    'LEU': ['N','CA','C','O','CB','CG','CD1','CD2'],
    'VAL': ['N','CA','C','O','CB','CG1','CG2'],
    'ILE': ['N','CA','C','O','CB','CG1','CG2','CD1'],
}
STANDARD_ATOMS = ['N','CA','C','O','CB','CG','CG1','CG2','OG','OG1',
                  'CD','CD1','CD2','CE','CE1','CE2','CZ','NZ','OE1','OE2',
                  'ND1','ND2','NE','NE1','NE2','NH1','NH2','OH','SD','SG']
atom_to_idx = {a:i for i,a in enumerate(STANDARD_ATOMS)}

def residue_to_atom14(res_name, coords_dict, n_atoms=14):
    """Convert residue coords to Atom14 format (fixed 14-slot tensor)."""
    atoms_present = RESIDUE_ATOMS.get(res_name, ['N','CA','C','O'])
    atom14 = torch.zeros(n_atoms, 3)
    atom14_mask = torch.zeros(n_atoms)
    for slot, atom_name in enumerate(atoms_present[:n_atoms]):
        if atom_name in coords_dict:
            atom14[slot] = torch.tensor(coords_dict[atom_name])
            atom14_mask[slot] = 1.0
    return atom14, atom14_mask

# Example
coords = {'N': [10.0, 20.0, 30.0], 'CA': [11.2, 20.5, 30.8],
          'C': [12.1, 21.2, 30.1], 'O': [12.5, 22.3, 30.5], 'CB': [11.0, 20.0, 32.2]}
atom14, mask = residue_to_atom14('ALA', coords)
print(f"Atom14 tensor: {atom14.shape}")
print(f"Mask: {mask[:6]}")
print(f"Present atoms: {mask.sum().int()}")
print(f"N coords: {atom14[0]}")
print(f"CB coords: {atom14[4]}")

## Atom14 / Atom37 All-Atom Representation
### AF3 operates on every heavy atom, not just Cα

In [ ]:
import torch
import numpy as np

def build_msa_features(msa_sequences, query_seq, max_seqs=512):
    """Build MSA feature tensor from aligned sequences.

    Returns:
      msa_feat: (S, L, c) where c includes one-hot AA + deletion features
      msa_mask: (S, L) — 1 if residue present, 0 if gap
    """
    AA = 'ACDEFGHIKLMNPQRSTVWY-'  # 20 AAs + gap
    aa2i = {a:i for i,a in enumerate(AA)}

    S = min(len(msa_sequences), max_seqs)
    L = len(query_seq)
    c = len(AA)

    msa_oh = torch.zeros(S, L, c)
    msa_mask = torch.zeros(S, L)
    deletion_count = torch.zeros(S, L)

    for s, seq in enumerate(msa_sequences[:S]):
        del_count = 0
        pos = 0
        for ch in seq:
            if ch.islower():  # lowercase = deletion
                del_count += 1
            else:
                if pos < L:
                    idx = aa2i.get(ch.upper(), aa2i['-'])
                    msa_oh[s, pos, idx] = 1.0
                    msa_mask[s, pos] = 1.0 if ch != '-' else 0.0
                    deletion_count[s, pos] = del_count
                    del_count = 0
                    pos += 1

    # Deletion mean feature
    del_feat = 2/np.pi * torch.arctan(deletion_count / 3)
    msa_feat = torch.cat([msa_oh, del_feat.unsqueeze(-1)], dim=-1)
    return msa_feat, msa_mask

query = 'ACDEFGHIKLM'
msa = [query, 'ACDEFGHILM-', 'ACDEFGaHIKLM', 'AaDEFGHIKLM']
feat, mask = build_msa_features(msa, query)
print(f"MSA features: {feat.shape}  (S, L, 22)")
print(f"MSA mask:     {mask.shape}")
print(f"Non-gap fraction: {mask.mean():.2%}")

## MSA Featurization
### Building the MSA feature tensor from sequence alignments

In [ ]:
import torch

# AF3 tokenization: unified sequence for protein + DNA + RNA + ligand
def tokenize_af3(protein_seq=None, dna_seq=None, rna_seq=None,
                  ligand_smiles=None):
    """
    AF3 novel feature: all molecule types in one token sequence.
    - Protein: 1 token per residue (Cα-based)
    - DNA/RNA: 1 token per nucleotide
    - Ligand:  1 token per HEAVY ATOM (from CCD)
    """
    tokens = []
    token_types = []

    if protein_seq:
        for aa in protein_seq:
            tokens.append(aa)
            token_types.append('protein_residue')

    if dna_seq:
        for nt in dna_seq:
            tokens.append(nt)
            token_types.append('dna_nucleotide')

    if rna_seq:
        for nt in rna_seq:
            tokens.append(nt)
            token_types.append('rna_nucleotide')

    if ligand_smiles:
        # Count heavy atoms (simplified: count non-H chars)
        import re
        heavy_atoms = re.findall(r'[BCNOPS]|Cl|Br|F|I', ligand_smiles)
        for atom in heavy_atoms:
            tokens.append(atom)
            token_types.append('ligand_atom')

    return tokens, token_types

# Example: TCR-pMHC complex
tokens, types = tokenize_af3(
    protein_seq='ACDEFGHIKLM',          # TCR chain
    dna_seq=None,
    rna_seq=None,
    ligand_smiles='CC(=O)Nc1ccc(O)cc1'  # paracetamol (small molecule)
)
print(f"Total tokens: {len(tokens)}")
from collections import Counter
type_counts = Counter(types)
for t, n in type_counts.items():
    print(f"  {t}: {n} tokens")
print()
print("Key insight: ligands have 1 token per ATOM (vs 1 per residue for protein)")
print("This allows AF3 to model all-atom structures without residue-level abstractions")

## Multi-Biomolecule Tokenization
### AF3's core novelty: protein + DNA + RNA + ligand in one model

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class SimpleDiffusion(nn.Module):
    """Simplified denoising diffusion for 3D coordinates (AF3-inspired).

    AF3 uses SE(3) diffusion on atom coordinates.
    Here we implement Euclidean diffusion for illustration.
    """
    def __init__(self, n_steps=200, beta_min=0.01, beta_max=0.02):
        super().__init__()
        self.n_steps = n_steps
        betas = torch.linspace(beta_min, beta_max, n_steps)
        alphas = 1 - betas
        self.alpha_bar = torch.cumprod(alphas, dim=0)
        self.betas = betas

    def forward_process(self, x0, t):
        """Add noise: x_t = sqrt(ᾱ_t) * x0 + sqrt(1-ᾱ_t) * ε"""
        ab = self.alpha_bar[t]
        eps = torch.randn_like(x0)
        x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
        return x_t, eps

    def reverse_step(self, x_t, pred_eps, t):
        """One denoising step: x_{t-1} from x_t and predicted noise"""
        b = self.betas[t]
        ab = self.alpha_bar[t]
        ab_prev = self.alpha_bar[t-1] if t > 0 else torch.tensor(1.0)
        x0_pred = (x_t - (1-ab).sqrt() * pred_eps) / ab.sqrt()
        x0_pred = x0_pred.clamp(-5, 5)
        mean = ab_prev.sqrt() * b / (1-ab) * x0_pred +                (1-b).sqrt() * (1-ab_prev) / (1-ab) * x_t
        var = b * (1 - ab_prev) / (1 - ab)
        return mean + var.sqrt() * torch.randn_like(x_t) if t > 0 else mean

torch.manual_seed(42)
diff = SimpleDiffusion(n_steps=200)
x0 = torch.randn(20, 3)  # 20 atoms

# Forward: add noise at t=100
x_noisy, eps = diff.forward_process(x0, t=100)
print(f"x0 std: {x0.std():.3f}")
print(f"x_t (t=100) std: {x_noisy.std():.3f}")

# Signal ratio at different timesteps
for t in [0, 50, 100, 150, 199]:
    snr = diff.alpha_bar[t].sqrt() / (1-diff.alpha_bar[t]).sqrt()
    print(f"  t={t:3d}: SNR={snr.item():.4f}")

## Diffusion Module
### AF3's structure generation via denoising diffusion

In [ ]:
import torch
import numpy as np

def spatial_crop(atoms_coords, n_crop=384, interface_residues=None):
    """AF3 spatial cropping: select contiguous spatial region for training.

    Strategy: pick a center atom/residue, include all within radius R.
    For complexes: bias toward interface residues.
    """
    n = len(atoms_coords)
    if n <= n_crop:
        return list(range(n)), atoms_coords

    # Pick center (biased to interface if provided)
    if interface_residues and len(interface_residues) > 0:
        center_idx = np.random.choice(interface_residues)
    else:
        center_idx = np.random.randint(n)

    center = atoms_coords[center_idx]
    dists = np.linalg.norm(atoms_coords - center, axis=-1)
    sorted_idx = np.argsort(dists)
    selected = sorted(sorted_idx[:n_crop].tolist())
    return selected, atoms_coords[selected]

np.random.seed(42)
# Simulate a 600-residue complex (protein-protein)
all_coords = np.random.randn(600, 3) * 20
interface = list(range(250, 350))  # residues near interface

selected_idx, cropped = spatial_crop(all_coords, n_crop=384, interface_residues=interface)
interface_in_crop = sum(1 for i in selected_idx if i in interface)
print(f"Original: {len(all_coords)} residues")
print(f"Cropped:  {len(cropped)} residues")
print(f"Interface residues in crop: {interface_in_crop}/{len(interface)} ({interface_in_crop/len(interface):.0%})")
print("Spatial crop ensures the model sees compact, biologically meaningful regions")

## Spatial and Interface Cropping
### Training strategy for large complexes

In [ ]:
import torch

def violation_loss(pred_coords, bond_lengths=None, clash_threshold=1.8, eps=1e-8):
    """AF3 violation loss: penalize bond length and steric clash violations.

    Two components:
    1. Bond length violation: |predicted_dist - ideal_dist| > tolerance
    2. Clash violation: predicted CA-CA distance < clash_threshold
    """
    n = len(pred_coords)
    total_loss = torch.tensor(0.0)

    # 1. Bond length violations (consecutive CA-CA ~3.8Å)
    if n > 1:
        ca_dists = torch.norm(pred_coords[1:] - pred_coords[:-1], dim=-1)
        ideal = 3.8
        tolerance = 0.5
        bond_viol = torch.clamp(torch.abs(ca_dists - ideal) - tolerance, min=0)
        total_loss = total_loss + bond_viol.mean()
        print(f"Bond lengths (mean ± std): {ca_dists.mean():.2f} ± {ca_dists.std():.2f} Å")
        print(f"Bond violations (> {tolerance}Å from {ideal}Å): {(bond_viol>0).sum()}/{n-1}")

    # 2. Steric clashes (all CA-CA < clash_threshold)
    dists = torch.cdist(pred_coords, pred_coords)
    dists.fill_diagonal_(float('inf'))
    clashes = torch.clamp(clash_threshold - dists, min=0)
    clash_loss = clashes.sum() / max(n, 1)
    total_loss = total_loss + clash_loss
    n_clashes = (dists < clash_threshold).sum().item() // 2
    print(f"Steric clashes (< {clash_threshold}Å): {n_clashes}")

    return total_loss

torch.manual_seed(42)
# Good prediction: near-ideal bond lengths
L = 20
good_coords = torch.cumsum(torch.randn(L, 3).clamp(-1,1) * 3.8 + torch.tensor([3.8, 0., 0.]), dim=0)
print("Good prediction:")
loss = violation_loss(good_coords)
print(f"Violation loss: {loss.item():.4f}\n")

# Bad prediction: random coords (many clashes)
bad_coords = torch.randn(L, 3) * 2
print("Bad prediction:")
loss = violation_loss(bad_coords)
print(f"Violation loss: {loss.item():.4f}")

## Violation Loss
### Bond geometry enforcement in AF3

In [ ]:
# Module 07/02 Resources
print("=" * 60)
print("MODULE 07/02 — AF3 Data Pipeline — Resources")
print("=" * 60)

topics = {
    "Data Formats": [
        "mmCIF: crystallographic information format (supersedes PDB)",
        "CCD: Chemical Component Dictionary (~100k small molecules)",
        "MSA: Multiple Sequence Alignment (A3M, Stockholm formats)",
        "SDF/MOL2: small molecule 2D structure formats",
    ],
    "Key Tools": [
        "Gemmi: fast mmCIF parser (C++/Python)",
        "Biopython: Bio.PDB for structure parsing",
        "rdkit: SMILES parsing, atom typing, conformer generation",
        "HH-suite / Jackhmmer: MSA generation tools",
    ],
    "AF3 Data Pipeline Steps": [
        "1. mmCIF parsing → residue/atom coordinates",
        "2. MSA search (Jackhmmer vs UniRef90, JackHMMer vs MGnify)",
        "3. Template search (HHsearch vs PDB)",
        "4. Atom14/37 featurization",
        "5. Tokenization (protein+DNA+RNA+ligand unified)",
        "6. Spatial/interface cropping for training",
    ],
}
for cat, items in topics.items():
    print(f"\n{cat}:")
    for item in items:
        print(f"  * {item}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **Atom14 representation**: 14 max heavy atoms per residue (glycine has 4, tryptophan has 14) — compact fixed-size tensor
- **MSA coevolution**: Direct Coupling Analysis (DCA), pseudolikelihood maximization — correlated mutations signal contacts
- **Relative position encoding**: 64 distance bins (log-spaced up to 32 residues apart) — encodes sequence separation
- **mmCIF format**: CIF (Crystallographic Information File) — replaced PDB format in 2019 for structures >62 chains or 99,999 atoms
- **Data augmentation**: random cropping, chain permutation, dropout of MSA rows during training

### 2️⃣ Must-Have Popular Resources
- 📘 MSA Transformer (Rao 2021) — coevolution via axial attention over multiple sequence alignments
- 📘 Bioinformatics Algorithms ch. 6+ (Compeau & Pevzner) — free online; sequence alignment and MSA theory
- 🎓 RCSB PDB documentation — mmCIF dictionary, structure validation, data access APIs
- ⭐ GitHub [biopython/biopython](https://github.com/biopython/biopython) 4.5k★ — mmCIF parser, structure objects
- ⭐ GitHub [biotite-dev/biotite](https://github.com/biotite-dev/biotite) 0.5k★ — faster, NumPy-native structure IO

### 3️⃣ Practicals — Hands-On
- 💻 Parse a real mmCIF file with BioPython — extract backbone (N, CA, C, O) atoms into NumPy arrays
- 💻 Build an Atom14 tensor for a 50-residue chain — verify shape (50, 14, 3) with correct atom masks
- 💻 Implement MSA diversity sampling — keep top-K sequences by Hamming distance
- 🤗 HuggingFace: [OpenProteinSet](https://huggingface.co/datasets/OpenProteinSet) — 270TB MSAs used to train AF2
- 🤗 HuggingFace: [plinder-org/plinder](https://huggingface.co/datasets/plinder-org/plinder) — protein–ligand structures

### 4️⃣ Real-World Problems
- 🏭 Every structure prediction company (computational biology ML teams, Schrödinger, Relay) has a custom mmCIF featurization pipeline
- 📊 Dataset: [RCSB PDB mmCIF archive](https://files.rcsb.org/pub/pdb/data/structures/) — updated weekly
- 📊 Dataset: [OpenProteinSet on HuggingFace](https://huggingface.co/datasets/OpenProteinSet) — pre-computed MSAs
- 🔬 Application: Temporal train/test split — proteins released after training cutoff avoid data leakage

### 5️⃣ Interview Tips
- ❓ Must know: Atom14 vs Atom37 — why use 14? (saves memory; Atom37 pads all residues to tryptophan's atom count)
- ❓ Must know: How do you do train/test split for proteins? (sequence identity <30% + date cutoff, NOT random split)
- ❓ Must know: Why 64 distance bins for relative position encoding? (captures local + long-range; log-spacing handles 1–32 separation)
- ⚠️ Common mistake: Using random splits — sequence similarity causes severe data leakage in structure prediction
- 💡 Pro tip: Mention mmCIF's handling of multi-chain complexes and crystallographic symmetry when discussing data pipelines

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [AlphaFold Database](https://alphafold.ebi.ac.uk/) — 200M predicted structures on EBI, HuggingFace mirror available
- 🔥 Trending: [ESMAtlas](https://esmatlas.com/) — Meta AI 700M predicted MGnify structures
- 🔥 Trending: [PLINDER benchmark](https://www.plinder.sh/) — standardized protein–ligand interaction dataset for fair evaluation
- 🚀 Build this: A featurization pipeline that takes a raw PDB ID, downloads mmCIF, extracts Atom14 tensors, and writes a PyTorch dataset

## Interview Q&A — AF3 Data Pipeline

**Q: What is Atom14 representation and why does AlphaFold use it?**
A: Atom14: represent each residue with 14 atom positions (padded with zeros for residues with fewer atoms). Glycine has 4 heavy atoms, Tryptophan has 14. Using a fixed-size 14-vector per residue enables batched tensor operations. Alternative: sparse list of atoms (variable length, harder to batch).

**Q: Why does AlphaFold use MSA (multiple sequence alignment) as input?**
A: Evolutionary co-variation in MSA reveals structural contacts. If position i and position j co-evolve across species, they're likely in contact in 3D (they need to be compatible). AlphaFold2's Evoformer processes MSA depth × sequence length matrices to extract these coevolution signals. AF3 uses a simplified Pairformer instead.

**Q: What is the mmCIF format and how does it differ from PDB format?**
A: mmCIF (macromolecular Crystallographic Information File): structured key-value format, handles large complexes, supports all atom types, no line-length limit. PDB (legacy): column-fixed format, max 100,000 atoms, struggles with large complexes. AF3 uses mmCIF. Biopython's `MMCIFParser` reads it.

**Q: How does chemical component dictionary (CCD) featurize ligands?**
A: CCD: standardized library of ~60,000 small molecule components with SMILES, atom types, bond orders. For AF3: tokenize ligand atoms individually (unlike proteins which tokenize residues). Each ligand atom gets element type, formal charge, is_in_ring, is_aromatic as features. This enables all-atom structure prediction.

### AF3 Data Pipeline — Time Complexity
| Step | Time | Space | Notes |
|------|------|-------|-------|
| mmCIF parsing | O(n_atoms) | O(n_atoms) | ~10ms for typical protein |
| Atom14 construction | O(n_residues) | O(n_residues × 14) | — |
| MSA construction (jackhmmer) | O(n_seqs × L × L) | O(n_seqs × L) | Hours for deep MSAs |
| Template search (HHsearch) | O(L²) per template | O(n_templates × L) | — |
| Pairformer (n_res) | O(n_res² × d) | O(n_res²) | Dominant cost |
| Triangle attention | O(n_res² × n_res) | O(n_res²) | Per-layer |

In [ ]:
# AF3 DATA PIPELINE — Feature Summary and Integration Quiz
import numpy as np

print("AF3 INPUT FEATURES CHEAT SHEET")
print("=" * 60)

features = [
    ("token_index",       "(N,)",       "integer residue/atom index"),
    ("residue_index",     "(N,)",       "residue sequence position"),
    ("token_type",        "(N,)",       "0=protein, 1=DNA, 2=RNA, 3=ligand"),
    ("target_feat",       "(N, 22)",    "one-hot AA type + special tokens"),
    ("msa_feat",          "(D, N, 49)", "D=depth; 49 profile features per position"),
    ("extra_msa_feat",    "(E, N, 25)", "E=extra depth; compressed MSA features"),
    ("pair_dist_bins",    "(N, N, 64)", "log-binned Cbeta-Cbeta distances (recycling)"),
    ("template_pair",     "(T, N, N, 88)", "T templates; geometric pair features"),
    ("atom14_gt_exists",  "(N, 14)",    "which atoms are present in ground truth"),
    ("atom14_gt_pos",     "(N, 14, 3)", "3D coordinates for up to 14 heavy atoms/residue"),
]

print(f"{'Feature':<22} {'Shape':<22} {'Description'}")
print("-" * 80)
for name, shape, desc in features:
    print(f"  {name:<20} {shape:<22} {desc}")

print()
print("KEY CONCEPT: Atom14 vs Atom37")
print("-" * 40)
print("  Atom37: 37 possible atom types in PDB ATOM records (sparse)")
print("  Atom14: max 14 heavy atoms per residue (glycine=4, trp=14)")
print("  AF3 uses Atom14 for efficiency: fixed N x 14 x 3 tensor")
print("  Ligands: each atom becomes its own 'token' (atom-level granularity)")

print()
print("INTEGRATION QUIZ — answer without looking:")
quiz = [
    ("Why is msa_feat shape (D, N, 49) not (D, N, 20)?",
     "49 = 20 AA types + gap + 21 deletion features + profile stats"),
    ("What does pair_dist_bins encode and when is it used?",
     "Cbeta distances from PREVIOUS recycling iter → passed back as input"),
    ("Why does template_pair have shape (T, N, N, 88)?",
     "T=4 templates max; 88 features encode phi/psi, dist bins, exists mask"),
    ("What changes in data pipeline when input includes a ligand?",
     "Ligand atoms become separate tokens; CCD looked up for atom types/bonds"),
]
for i, (q, a) in enumerate(quiz, 1):
    print(f"  Q{i}: {q}")
    print(f"       ANS: {a}")
    print()

print("Self-assessment: can you explain each answer in your own words?")
print("If not, revisit the AF3 supplementary (Methods sections 1-3).")

# 🌍 Real World Problems — AF3 Data Pipeline

## Problem 1 — Parse a Real mmCIF Structure and Build Atom14
**Dataset/Source**: RCSB PDB — any mmCIF file (e.g. 7CWY: SARS-CoV-2 Spike)
**GitHub**: [github.com/google-deepmind/alphafold](https://github.com/google-deepmind/alphafold/blob/main/alphafold/data/mmcif_parsing.py)
**Hugging Face**: [huggingface.co/datasets/chandar-lab/ProteinBench](https://huggingface.co/datasets/chandar-lab/ProteinBench)
**Skills**: mmCIF parsing, Atom14 representation, all-atom structure handling

```python
import urllib.request

def fetch_mmcif(pdb_id):
    url = f'https://files.rcsb.org/download/{pdb_id}.cif'
    with urllib.request.urlopen(url) as r:
        return r.read().decode()

# YOUR TASK:
# 1. mmcif = fetch_mmcif('6VXX')  # SARS-CoV-2 spike trimer (181K atoms)
# 2. atoms = parse_mmcif(mmcif)
# 3. Group atoms by residue: {(chain, res_num, res_name): {atom_name: coords}}
# 4. For each residue, call residue_to_atom14_coords() to get (14,3) + (14,) mask
# 5. Stack all residues: atom14_coords (N, 14, 3), atom14_mask (N, 14)
# 6. Compare: how many atoms in atom14 vs just Cα?
# Key insight: TRP has 14 atoms in atom14 vs 1 in Cα-only
# AF3 uses ALL of these for FAPE loss computation
```

**Real impact**: AF3's all-atom FAPE captures side-chain packing accuracy — critical for drug binding prediction where the ligand must fit into the correct rotamer configuration.

---

## Problem 2 — Build MSA from UniProt via MMseqs2
**Dataset/Source**: SARS-CoV-2 spike protein + 50 homologs from UniRef50
**GitHub**: [github.com/soedinglab/MMseqs2](https://github.com/soedinglab/MMseqs2) — fast MSA search
**Hugging Face**: [huggingface.co/datasets/tattabio/bac_16s_rrna](https://huggingface.co/datasets/tattabio/bac_16s_rrna)
**Skills**: MSA featurization, deletion matrix, depth analysis

```python
# Using ColabFold's MMseqs2 API (free, no installation)
import urllib.request, json

def colabfold_msa(sequence, mode='env'):
    """Fetch MSA from ColabFold server using MMseqs2."""
    # ColabFold batch API
    url = 'https://api.colabfold.com/ticket/msa'
    data = json.dumps({'q': f'>query\n{sequence}', 'mode': mode}).encode()
    req = urllib.request.Request(url, data=data,
                                  headers={'Content-Type': 'application/json'})
    # This submits a job; real usage requires polling for result
    # See: github.com/sokrypton/ColabFold for full API usage
    pass

# YOUR TASK:
# 1. Take the first 100 AA of SARS-CoV-2 spike: MFVFLVLLPLVSSQCVNLTTRTQLPPAYTN...
# 2. Submit to ColabFold API (or use local MMseqs2)
# 3. Parse the returned MSA (A3M format)
# 4. Call featurize_msa() on the result
# 5. Analyze: how does MSA depth correlate with pLDDT in AF3 predictions?
# Published result: AF3 pLDDT correlation with Neff (effective MSA depth) is r=0.71

# Quick simulation to see the effect:
for depth in [1, 8, 64, 512, 2048]:
    # More MSA sequences = better co-evolution signal
    sim_msa = ['ACDEFGHIKL' for _ in range(depth)]
    feat, mask, _ = featurize_msa(sim_msa)
    coverage = mask.mean()
    print(f'MSA depth {depth:5d}: coverage={coverage:.3f}, '
          f'profile entropy ~{max(0, 2.5 - depth/1000):.2f} bits')
```

**Real impact**: AF3 MSA processing caps at 16,384 sequences for efficiency; for viral proteins with 100K+ homologs (e.g. HIV protease), subsampling strategy dramatically affects prediction quality.

---

## Problem 3 — Sample Multiple Conformations with Diffusion
**Dataset/Source**: Calmodulin (PDB 1CLL — open) vs (PDB 1CDL — closed) — two conformations
**GitHub**: [github.com/google-deepmind/alphafold3](https://github.com/google-deepmind/alphafold3) — official AF3 inference
**Kaggle**: [Protein Structure Prediction](https://www.kaggle.com/competitions/novozymes-enzyme-stability-prediction)
**Skills**: Diffusion noise schedule, conformation sampling, structural diversity

```python
def sample_conformations(x_true, denoising_fn, n_samples=5, sigma_final=0.1):
    """
    Sample multiple conformations by running diffusion from different noise seeds.
    AF3 generates 5 structures by default, ranked by confidence.
    """
    conformations = []
    for i in range(n_samples):
        np.random.seed(i)
        # Start from high noise (sigma=20Å)
        x_noisy, _ = forward_diffusion(x_true, sigma=20.0)
        
        # Progressively denoise (simplified T=10 steps)
        sigmas = np.exp(np.linspace(np.log(20.0), np.log(sigma_final), 10))
        x = x_noisy
        for sigma in sigmas:
            # Perfect denoiser oracle (in AF3: neural network does this)
            x_pred = denoising_fn(x, sigma)
            # Add back small noise for next step (DDIM-style)
            if sigma > sigma_final:
                x, _ = forward_diffusion(x_pred, sigma=sigma*0.7)
            else:
                x = x_pred
        conformations.append(x)
    return conformations

# Perfect oracle denoiser: returns true structure (for testing)
def oracle_denoiser(x_noisy, sigma):
    return x_true_calmodulin + np.random.normal(0, sigma*0.1, x_true_calmodulin.shape)

# Simulate calmodulin open/closed conformations
np.random.seed(42)
N_res = 148  # calmodulin has 148 residues
x_true_calmodulin = np.random.randn(N_res, 3) * 8  # mock coordinates

conformations = sample_conformations(x_true_calmodulin, oracle_denoiser, n_samples=5)
rmsds = [np.sqrt(np.mean((c - x_true_calmodulin)**2)) for c in conformations]
print('5 sampled conformations RMSD from reference:')
for i, rmsd in enumerate(rmsds):
    print(f'  Conformation {i+1}: RMSD = {rmsd:.2f}Å')
print('\nAF3: rank by confidence (pLDDT), return top-1 or full ensemble')
print('Use case: calmodulin has open/closed forms — sample both with diffusion')
```

**Real impact**: AF3 generates 5 structural hypotheses per prediction (ranked by confidence); for drug discovery, sampling multiple conformations of a flexible binding site improves docking hit rates by 30-40%.

---

## Resources
| Resource | URL | Purpose |
|----------|-----|---------|
| AF3 Official Code | github.com/google-deepmind/alphafold3 | Official AF3 inference code |
| OpenFold | github.com/aqlaboratory/openfold | Reproducible PyTorch AF2/AF3 |
| MMseqs2 | github.com/soedinglab/MMseqs2 | Fast MSA search for AF3 input |
| ColabFold | github.com/sokrypton/ColabFold | Free AF2/AF3 inference + MSA |
| RCSB PDB mmCIF | files.rcsb.org/download/6VXX.cif | mmCIF format examples |
| AF3 Paper | doi.org/10.1038/s41586-024-07487-w | Abramson et al. Nature 2024 |
| Kaggle Novozymes | kaggle.com/competitions/novozymes-enzyme-stability-prediction | $50K stability competition |

## Mastery Check

On a first pass, success means you can:
1. explain what this notebook's component does in the larger AF3 pipeline
2. describe why geometry matters here
3. explain at least one confidence or training metric in words
4. decide whether you should revisit this notebook later for deeper implementation work


## 🐛 Debug Exercise — AF3 Data Pipeline

Find and fix the **3 bugs** in the MSA feature encoding below.

**Expected correct output:**
```
One-hot encoding dim: 21  (20 amino acids + 1 gap token = 21 total)
Template mask correctly zero for missing residues
Residue index 0 maps to PDB residue 1 (1-indexed PDB convention)
```

<details>
<summary>Show answer</summary>

**Bug 1 — Wrong one-hot dimension:** The alphabet has 20 standard amino acids + 1 gap '-' = 21
total tokens, so `np.zeros((L, 21))` is correct. The bug is using 20, which drops the gap.

**Bug 2 — Template not masked for missing residues:** Residues with coordinates (0,0,0) are
placeholder "missing" residues. Fix: `mask = np.any(template_coords != 0, axis=-1)` and
zero out features where `mask == False`.

**Bug 3 — Off-by-one in residue indexing:** PDB files use 1-based residue numbering.
Converting to 0-based internally without tracking the offset causes misalignment with
structural coordinates. Fix: store `pdb_resnum - 1` as the 0-based index and use that
consistently, or keep 1-based throughout.
</details>


In [ ]:
# DEBUG EXERCISE — AF3 data pipeline feature encoding, find and fix 3 bugs
import numpy as np

AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'  # 20 standard amino acids
GAP = '-'
FULL_ALPHABET = AA_ALPHABET + GAP     # 21 tokens total

def encode_msa(sequences):
    """One-hot encode a list of aligned sequences."""
    L = len(sequences[0])
    depth = len(sequences)

    # BUG 1: one-hot dimension is 20, but alphabet has 21 tokens (20 AA + gap)
    # gap token '-' will be silently ignored / misassigned
    one_hot = np.zeros((depth, L, 20), dtype=np.float32)   # should be 21

    for s_idx, seq in enumerate(sequences):
        for pos, aa in enumerate(seq):
            if aa in FULL_ALPHABET:
                aa_idx = FULL_ALPHABET.index(aa)
                if aa_idx < 20:   # gap (idx=20) never gets encoded — silent bug
                    one_hot[s_idx, pos, aa_idx] = 1.0
    return one_hot

msa = ["ACDE-", "AC-EF", "ACDEG"]
encoded = encode_msa(msa)
print(f"One-hot encoding shape: {encoded.shape}  (depth=3, L=5, channels=?)")
print(f"Expected channels: 21, got: {encoded.shape[2]}")

# BUG 2: template features not masked for missing residues
template_coords = np.array([
    [1.2, 3.4, 5.6],
    [0.0, 0.0, 0.0],   # missing residue — coordinates are placeholder zeros
    [2.1, 4.5, 6.7],
])
# Missing residues should have their features zeroed out (masked)
# The fix: mask = np.any(template_coords != 0, axis=-1)
template_features = np.random.randn(3, 64)  # some feature vector per residue
# BUG: no masking applied — missing residue features contain random noise
print(f"\nTemplate feature row 1 (missing residue, should be zeros): {template_features[1, :4]}")

# BUG 3: residue numbering off-by-one
pdb_resnums = [1, 2, 3, 4, 5]   # PDB uses 1-based numbering
# Incorrectly storing as 0-indexed without offset tracking
internal_indices = pdb_resnums   # should be [r - 1 for r in pdb_resnums]
print(f"\nPDB resnum 1 stored as internal index: {internal_indices[0]}")
print("Expected internal index for PDB resnum 1: 0 (0-based), got: 1 (off by one)")


## ✏️ Exercise: Raw Sequence → AF3 Input Tensors

Before AF3 can run, a raw protein sequence string must be converted into the four input tensors the model expects.

**Your task:** implement `sequence_to_af3_inputs(seq: str)` that returns a dictionary with four tensors:

| Key | Shape | Description |
|-----|-------|-------------|
| `token_idx` | `(N,)` int | Integer index of each amino acid in `AMINO_ACIDS` |
| `residue_features` | `(N, 20)` float | One-hot encoding of each residue |
| `relative_position` | `(N,)` int | 0-based position index |
| `chain_id` | `(N,)` int | All zeros (single chain) |

**Test:** call the function on `'KVLGSGAFGT'` and verify each tensor's shape.

## Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ModuleNotFoundError: No module named 'X'` | Package not installed | Run `!pip install X` in a cell, then restart kernel |
| `CUDA out of memory` | GPU RAM exceeded | Reduce batch size, or add `torch.cuda.empty_cache()` |
| `RuntimeError: Expected all tensors on same device` | Mixed CPU/GPU tensors | Add `.to(device)` after creating each tensor |
| `ValueError: shapes not aligned` | Matrix dimension mismatch | Print `tensor.shape` before the operation to debug |
| `KeyError` in DataFrame | Column name wrong or missing | Print `df.columns` to see exact column names |
| `IndexError: index out of range` | Loop or slice exceeds sequence length | Print `len(sequence)` and check your index |
| Kernel dies silently | Memory overflow (RAM) | Restart kernel, reduce data size, use generators |
| `UserWarning: No GPU found` | CUDA not available | Add `device = 'cuda' if torch.cuda.is_available() else 'cpu'` |

In [ ]:
import torch
import torch.nn.functional as F

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'  # 20 standard amino acids
AA_TO_IDX   = {aa: i for i, aa in enumerate(AMINO_ACIDS)}


def sequence_to_af3_inputs(seq: str) -> dict:
    """
    Convert a raw protein sequence string to AF3 input tensors.

    Args:
        seq: Protein sequence string, e.g. 'ACDEFG'
    Returns:
        dict with keys: token_idx, residue_features,
                        relative_position, chain_id
    """
    N = len(seq)

    # TODO 1: token_idx — map each character to its integer index
    # Use AA_TO_IDX; treat unknown amino acids as index 0.
    token_idx = None  # shape (N,), dtype=torch.long

    # TODO 2: residue_features — 20-dim one-hot per residue
    # Hint: F.one_hot(token_idx, num_classes=20).float()
    residue_features = None  # shape (N, 20), dtype=torch.float

    # TODO 3: relative_position — 0-based integer position index
    relative_position = None  # shape (N,), dtype=torch.long

    # TODO 4: chain_id — all zeros for a single chain
    chain_id = None  # shape (N,), dtype=torch.long

    return {
        'token_idx':         token_idx,
        'residue_features':  residue_features,
        'relative_position': relative_position,
        'chain_id':          chain_id,
    }


# ── Test ──────────────────────────────────────────────────────────────
test_seq = 'KVLGSGAFGT'
tensors = sequence_to_af3_inputs(test_seq)

expected_shapes = {
    'token_idx':         torch.Size([len(test_seq)]),
    'residue_features':  torch.Size([len(test_seq), 20]),
    'relative_position': torch.Size([len(test_seq)]),
    'chain_id':          torch.Size([len(test_seq)]),
}

all_pass = True
for key, expected_shape in expected_shapes.items():
    t = tensors.get(key)
    if t is None:
        print(f'  {key:<22}: NOT IMPLEMENTED')
        all_pass = False
    elif t.shape == expected_shape:
        print(f'  {key:<22}: {t.shape}  ✓')
    else:
        print(f'  {key:<22}: got {t.shape}, expected {expected_shape}  ✗')
        all_pass = False

if all_pass:
    print('\nAll shapes correct! Check one-hot sanity:')
    print(f'  Sum of residue_features per row: '
          f'{tensors["residue_features"].sum(dim=1).tolist()}')
    print('  (should all be 1.0)')

# ── SOLUTION (uncomment to check) ──────────────────────────────────────
# def sequence_to_af3_inputs(seq):
#     N = len(seq)
#     token_idx = torch.tensor(
#         [AA_TO_IDX.get(aa, 0) for aa in seq], dtype=torch.long)
#     residue_features = F.one_hot(token_idx, num_classes=20).float()
#     relative_position = torch.arange(N, dtype=torch.long)
#     chain_id = torch.zeros(N, dtype=torch.long)
#     return {'token_idx': token_idx, 'residue_features': residue_features,
#             'relative_position': relative_position, 'chain_id': chain_id}


## 🎤 Interview Questions

**Q1 (Easy):** What is an MSA and why is it important for structure prediction?
<details><summary>Answer</summary>
An MSA aligns a query protein with its evolutionary relatives, revealing which positions are conserved (functionally important) and which positions co-evolve (spatially close). AF2/AF3 use MSA features to build pair representations that encode co-evolutionary constraints. Deeper MSAs (more sequences) generally improve prediction accuracy because they provide stronger co-evolutionary signal.
</details>

**Q2 (Easy):** What is the difference between atom14 and atom37 representations?
<details><summary>Answer</summary>
atom14 uses 14 atom slots per residue, covering the standard heavy atoms of the 20 amino acids in a compact format. atom37 uses 37 slots to accommodate all possible atom types across all residue types (including modified residues and ligands). atom14 is memory-efficient for standard proteins; atom37 is more general. Conversion between them uses a residue-type-dependent mapping table.
</details>

**Q3 (Medium):** How does cropping work in AF3 training, and why is it necessary?
<details><summary>Answer</summary>
During training, a spatial or contiguous crop of fixed size (e.g., 384 residues) is selected from each protein complex. This is necessary because the pair representation is O(N²) in memory, and full complexes can be thousands of residues. Spatial cropping selects residues near a random center to maintain local structural context. Contiguous cropping takes a continuous chain segment. The crop size is a key hyperparameter balancing context vs. memory.
</details>

**Q4 (Medium):** Describe the data pipeline from a raw PDB/mmCIF file to model input tensors.
<details><summary>Answer</summary>
Steps: (1) Parse mmCIF to extract sequence, coordinates, chain IDs, entity mappings. (2) Search sequence databases (UniRef, BFD) to build MSA using JackHMMER/HHblits. (3) Search PDB for structural templates. (4) Featurize: convert sequences to residue type tokens, MSA to one-hot + deletion features, templates to distance/angle features, coordinates to atom14/atom37 arrays. (5) Crop to training size. (6) Assemble into feature dictionary with consistent tensor shapes. (7) Apply data augmentations (noise, masking). Each step has its own validation and error handling.
</details>

**Q5 (Hard):** AF3 handles proteins, nucleic acids, and small molecules in a unified pipeline. What are the key challenges in tokenizing and featurizing these different molecular types together?
<details><summary>Answer</summary>
Challenges: (1) Different vocabularies — 20 amino acids vs. 4 nucleotides vs. thousands of ligand atom types. Solution: unified token vocabulary with type embeddings. (2) Different atom counts — amino acids have ~14 heavy atoms, nucleotides ~23, ligands vary wildly. Solution: atom37 representation with masking, or per-atom featurization. (3) No MSA for ligands — co-evolutionary signal is protein-only. Solution: ligand features come from chemical properties (SMILES, atomic numbers, bond types). (4) Different coordinate conventions — proteins use backbone frames, ligands are defined by connectivity. (5) Chain/entity assignment for complexes with multiple copies. AF3 addresses these with a unified atom-level representation and entity-type embeddings.
</details>

## ✅ Notebook Complete

**You can now:**
- Trace the AF3 data pipeline from raw structure files to feature dictionaries
- Explain MSA generation, template search, and featurization
- Understand atom14/atom37 representations and cropping strategies
- Handle multi-entity complexes with proteins, nucleic acids, and ligands

**Next:** → `07_alphafold3_core/03_af3_evaluation.ipynb`